# 04 · Stacks and queues

After this notebook you can pick a stack or a queue from the shape of a problem, implement both correctly in Python (`list` and `collections.deque`), and solve the classics: brackets, min-stack, reverse Polish notation, the monotonic stack (daily temperatures, next greater element), a queue built from two stacks, sliding-window maximum with a monotonic deque, and a first breadth-first search.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Big-O and timing](01_big_o_and_timing.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 5 "Stacks & queues"

## Why this matters in interviews

- A stack models "most recent first" (nesting, undo, recursion); a queue models "first come, first served" (BFS, task queues, rate limiters). Interviewers expect you to pick the right one instantly.
- The **monotonic stack / deque** is the trick behind a whole family of "next greater", "span" and "window maximum" problems. It is O(n) despite a nested loop — and you must be able to say why.
- Python-specific: `list.pop(0)` is O(n). A production-quality answer uses `collections.deque`.

## What interviewers ask

- "Is this string of brackets valid?" (LeetCode 20)
- "Design a stack that also returns its minimum in O(1)." (LC 155)
- "Evaluate an expression in reverse Polish notation." (LC 150)
- "Daily temperatures: how many days until a warmer one?" (LC 739) · "Next greater element." (LC 496, 503)
- "Implement a queue using two stacks. What is the amortised cost?" (LC 232)
- "Maximum of every window of size k." (LC 239)
- "Shortest path in a grid." (BFS — LC 1091) · "Level-order traversal." (LC 102)
- Follow-ups: "Why not `list.pop(0)`?" · "Why is the monotonic stack O(n) with a nested loop?" · "What would change for DFS?"

In [ ]:
import operator
import random
import time
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

random.seed(0)
rng = random.Random(1)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})


def interleaved_times(calls, number=1, rounds=3):
    """Seconds per call for each fn in `calls`: best over rounds, alternating calls so background load hits all equally."""
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            t0 = time.perf_counter()
            for _ in range(number):
                call()
            best[i] = min(best[i], (time.perf_counter() - t0) / number)
    return best

## The pattern: LIFO vs FIFO

**In plain English:** a **stack** is a pile of plates — you add and remove at the top, so the *last* thing in is the *first* out (LIFO). A **queue** is a line at a counter — join at the back, leave from the front, *first* in, *first* out (FIFO).

```
 stack (LIFO) — a Python list                 queue (FIFO) — collections.deque
                                                 front                    back
   top -> | c |   <- push: s.append(x)             +-----+-----+-----+
          | b |      pop:  s.pop()        popleft <-|  a  |  b  |  c  |<- append
          | a |      peek: s[-1]                    +-----+-----+-----+
          +---+
```

**Recognise a stack when you see:** matching or nesting (brackets, tags), "most recent unmatched", undo / backspace, evaluating expressions, iterative DFS, "next greater / smaller" (monotonic stack).

**Recognise a queue when you see:** processing in arrival order, BFS / level order, shortest path in an unweighted graph, "in the last T seconds" (rate limiting), "maximum of every window" (monotonic deque).

**Templates:**

```python
stack = []
stack.append(x); top = stack[-1]; x = stack.pop()    # all O(1); check `if stack:` first

from collections import deque
q = deque()
q.append(x); x = q.popleft()                         # both O(1)

# monotonic stack: for every element, the next element to its right that is greater
answer, stack = [-1] * len(a), []                    # stack holds indices whose answer is still unknown
for i, x in enumerate(a):
    while stack and a[stack[-1]] < x:                # x answers every smaller element waiting on the stack
        answer[stack.pop()] = x
    stack.append(i)
```

Same input, two structures, opposite orders — plus `deque(maxlen=k)`, which silently drops from the far end (a "last k messages" conversation buffer is exactly this):

In [ ]:
stack, queue = [], deque()
for item in ["a", "b", "c"]:
    stack.append(item)
    queue.append(item)
print("stack pops :", [stack.pop() for _ in range(3)], " (LIFO)")
print("queue pops :", [queue.popleft() for _ in range(3)], " (FIFO)")

recent = deque(maxlen=3)                       # bounded: keeps only the newest 3
for msg in ["hi", "my name is Ana", "I like tea", "what is my name?", "and my drink?"]:
    recent.append(msg)
print("last 3 messages:", list(recent))
assert list(recent) == ["I like tea", "what is my name?", "and my drink?"]

### Why not `list.pop(0)`?

A list is a dynamic array: removing the *first* element shifts every other element one slot left — O(n). A `deque` is a doubly linked list of fixed-size blocks, so both ends are O(1). Measure one dequeue + enqueue on queues of growing size:

In [ ]:
sizes = [1_000, 10_000, 100_000, 400_000]
lists = {n: list(range(n)) for n in sizes}
deques = {n: deque(range(n)) for n in sizes}


def list_step(lst):
    return lambda: lst.append(lst.pop(0))        # dequeue from the front: shifts n - 1 pointers


def deque_step(dq):
    return lambda: dq.append(dq.popleft())       # O(1) at both ends


t_list = interleaved_times([list_step(lists[n]) for n in sizes], number=50)
t_deque = interleaved_times([deque_step(deques[n]) for n in sizes], number=5_000)
slope_list = np.polyfit(np.log10(sizes), np.log10(t_list), 1)[0]
slope_deque = np.polyfit(np.log10(sizes), np.log10(t_deque), 1)[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(sizes, np.array(t_list) * 1e6, marker="o", label=f"list: pop(0) + append (slope {slope_list:.1f})")
ax.loglog(sizes, np.array(t_deque) * 1e6, marker="o", label=f"deque: popleft() + append (slope {slope_deque:.1f})")
ax.set(title="One dequeue + enqueue: list is O(n), deque is O(1)", xlabel="queue length n (log scale)",
       ylabel="time per operation, µs (log scale)")
ax.minorticks_off()
ax.set_xticks(sizes, [f"{n:,}" for n in sizes])
ax.legend()
plt.show()
print(f"at n = {sizes[-1]:,}: list.pop(0) is {t_list[-1] / t_deque[-1]:,.0f}x slower than deque.popleft()")
assert slope_list > 0.6 and slope_deque < 0.3 and t_list[-1] > 20 * t_deque[-1]

Draining a queue of n items with `pop(0)` is therefore O(n²) in total — a classic hidden quadratic in BFS code. `deque` has one weakness to mention: indexing into the *middle* (`dq[i]`) is O(n).

## Problem 1 · Valid parentheses (LC 20)

**Statement:** a string of `()[]{}` is valid if every opener is closed by the same type in the right order. `"()[]{}"` → True, `"([)]"` → False, `"{[]}"` → True.

**Brute force:** repeatedly delete adjacent pairs `()`, `[]`, `{}` until nothing changes — O(n²). **Optimal:** push openers; each closer must match the *most recent* unmatched opener — the top of the stack.

In [ ]:
def is_valid_brute(s):
    while True:
        shorter = s.replace("()", "").replace("[]", "").replace("{}", "")
        if shorter == s:
            return s == ""
        s = shorter


def is_valid(s):
    closer_to_opener = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in closer_to_opener:
            if not stack or stack.pop() != closer_to_opener[ch]:   # nothing to close, or the wrong type
                return False
        else:
            stack.append(ch)
    return not stack                                              # leftover openers were never closed


for s, expected in [("()", True), ("()[]{}", True), ("(]", False), ("([)]", False), ("{[]}", True), ("", True),
                    ("(", False), (")", False), ("((", False), ("([]){}", True)]:
    assert is_valid(s) == expected == is_valid_brute(s), s
for _ in range(500):
    s = "".join(rng.choice("()[]{}") for _ in range(rng.randint(0, 10)))
    assert is_valid(s) == is_valid_brute(s)
print("is_valid: edge cases (empty, lone closer, leftovers) + 500 random strings agree with the brute force")

**Complexity:** O(n) time, O(n) space (all openers). **Traps:** popping an empty stack (`")"` alone) and forgetting to check leftovers at the end (`"(("`).

## Problem 2 · Min stack (LC 155)

**Statement:** a stack with `push`, `pop`, `top` and `get_min`, all O(1). **Brute force:** `min(stack)` on demand — O(n). **Optimal:** store, with every value, the minimum of the stack *up to and including it*. Popping restores the previous minimum automatically.

In [ ]:
class MinStack:
    def __init__(self):
        self._items = []                               # (value, min of everything at or below it)

    def push(self, x):
        current_min = min(x, self._items[-1][1]) if self._items else x
        self._items.append((x, current_min))

    def pop(self):
        return self._items.pop()[0]

    def top(self):
        return self._items[-1][0]

    def get_min(self):
        return self._items[-1][1]


ms = MinStack()
for x in [-2, 0, -3]:
    ms.push(x)
print("min after pushing -2, 0, -3:", ms.get_min())
ms.pop()
print("after one pop: top =", ms.top(), " min =", ms.get_min())
assert (ms.top(), ms.get_min()) == (0, -2)

for _ in range(200):                                    # random operations vs a plain list with min()
    ms, ref = MinStack(), []
    for _ in range(30):
        if ref and rng.random() < 0.4:
            assert ms.pop() == ref.pop()
        else:
            x = rng.randint(-9, 9)
            ms.push(x)
            ref.append(x)
        if ref:
            assert ms.get_min() == min(ref) and ms.top() == ref[-1]
print("MinStack: 200 random operation sequences agree with list + min()")

**Complexity:** O(1) per operation, O(n) space. (A space-saving variant keeps a second stack that only records new minimums.)

## Problem 3 · Evaluate reverse Polish notation (LC 150)

**Statement:** evaluate postfix tokens with `+ - * /`, where division **truncates toward zero**. `["2","1","+","3","*"]` → `(2 + 1) * 3` = 9.

**Optimal:** numbers go on a stack; an operator pops two operands, applies itself, and pushes the result. Two traps: the operand popped *first* is the **right-hand** one, and Python's `//` floors (`-7 // 2 == -4`) while this problem truncates (`-3`).

In [ ]:
def div_truncate(a, b):
    """Integer division that truncates toward zero (C/Java semantics). Python's // rounds toward minus infinity."""
    q = abs(a) // abs(b)
    return q if (a < 0) == (b < 0) else -q


OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": div_truncate}


def eval_rpn(tokens):
    stack = []
    for tok in tokens:
        if tok in OPS:
            right, left = stack.pop(), stack.pop()      # order matters for - and /
            stack.append(OPS[tok](left, right))
        else:
            stack.append(int(tok))                      # "-11" is a number, not an operator
    return stack.pop()


print("-7 // 2 =", -7 // 2, " but truncating division gives", div_truncate(-7, 2))
for tokens, expected in [(["2", "1", "+", "3", "*"], 9), (["4", "13", "5", "/", "+"], 6),
                         (["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"], 22),
                         (["-7", "2", "/"], -3), (["7", "-2", "/"], -3), (["3"], 3), (["0", "3", "/"], 0), (["5", "8", "-"], -3)]:
    assert eval_rpn(tokens) == expected, tokens
assert -7 // 2 == -4 and div_truncate(-7, 2) == -3
print("eval_rpn: all cases pass, including negative division")

**Complexity:** O(n) time, O(n) space. Stacks are how calculators, compilers and the Python interpreter itself evaluate expressions (CPython's bytecode runs on a value stack).

## Problem 4 · Daily temperatures — the monotonic stack (LC 739)

**Statement:** for each day, how many days until a warmer temperature (0 if never)? `[73,74,75,71,69,72,76,73]` → `[1,1,4,2,1,1,0,0]`.

**Brute force:** scan forward from every day — O(n²). **Optimal, in plain English:** keep a stack of days still *waiting* for a warmer day. Their temperatures are decreasing from bottom to top (a warmer day would already have answered them). When a new day is warmer than the top, it is the answer for that day — pop it and keep checking.

In [ ]:
def daily_temperatures_brute(temps):
    answer = [0] * len(temps)
    for i in range(len(temps)):
        for j in range(i + 1, len(temps)):
            if temps[j] > temps[i]:
                answer[i] = j - i
                break
    return answer


def daily_temperatures(temps):
    answer, waiting = [0] * len(temps), []        # waiting: indices, temperatures decreasing bottom -> top
    for i, t in enumerate(temps):
        while waiting and temps[waiting[-1]] < t:  # today is the first warmer day for these
            j = waiting.pop()
            answer[j] = i - j
        waiting.append(i)
    return answer


temps = [73, 74, 75, 71, 69, 72, 76, 73]
print(daily_temperatures(temps))
for case, expected in [(temps, [1, 1, 4, 2, 1, 1, 0, 0]), ([30, 40, 50, 60], [1, 1, 1, 0]), ([30, 60, 90], [1, 1, 0]),
                       ([], []), ([50, 50], [0, 0]), ([90, 80, 70], [0, 0, 0])]:
    assert daily_temperatures(case) == expected == daily_temperatures_brute(case), case
for _ in range(300):
    case = [rng.randint(30, 40) for _ in range(rng.randint(0, 20))]
    assert daily_temperatures(case) == daily_temperatures_brute(case)
print("daily_temperatures: edge cases (ties, strictly falling) + 300 random inputs agree with the brute force")

In [ ]:
wait = daily_temperatures(temps)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(temps)), temps, marker="o", ms=8, color="#898781", lw=1)
for i, w in enumerate(wait):
    if w:
        ax.annotate("", xy=(i + w, temps[i + w] + 0.4), xytext=(i, temps[i] + 0.4),
                    arrowprops=dict(arrowstyle="->", color=COLORS[0], lw=1.6, connectionstyle="arc3,rad=-0.35"))
    ax.text(i, min(temps) - 2.2, f"wait {w}", ha="center", fontsize=9, color="#52514e")
ax.set(title="Daily temperatures: each arrow points to the next warmer day", xlabel="day", ylabel="temperature",
       ylim=(min(temps) - 3, max(temps) + 5), xticks=range(len(temps)))
plt.show()

**Why O(n) with a nested loop:** every index is pushed once and popped at most once, so the `while` loop runs at most n times *in total* across the whole scan — an amortised argument. Check it by counting pops:

In [ ]:
def count_pops(temps):
    waiting, pops = [], 0
    for i, t in enumerate(temps):
        while waiting and temps[waiting[-1]] < t:
            waiting.pop()
            pops += 1
        waiting.append(i)
    return pops


n = 2_000
falling_then_hot = list(range(n - 1, 0, -1)) + [n]       # n-1 days wait, then one day answers them all at once
print(f"one day pops {count_pops(falling_then_hot):,} indices at once — still only {n - 1:,} pops in total")
worst_random = max(count_pops([rng.randint(0, 100) for _ in range(n)]) for _ in range(20))
print(f"most pops over 20 random inputs of length {n:,}: {worst_random:,}")
assert count_pops(falling_then_hot) == n - 1 and worst_random < n

## Problem 5 · Next greater element (LC 496 and circular LC 503)

**Statement:** for each element, the first element *to its right* that is larger, or −1. LC 496 asks it for a subset `nums1` of a distinct array `nums2`; LC 503 lets the search **wrap around** the end.

**Optimal:** the same monotonic stack. For LC 496, compute the answers for `nums2` once and look them up in a dict. For the circular version, scan the array **twice** (indices `i % n`), pushing only in the first pass.

In [ ]:
def next_greater(nums):
    answer, stack = [-1] * len(nums), []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        stack.append(i)
    return answer


def next_greater_element(nums1, nums2):               # LC 496
    greater = dict(zip(nums2, next_greater(nums2)))
    return [greater[x] for x in nums1]


def next_greater_circular(nums):                     # LC 503
    n = len(nums)
    answer, stack = [-1] * n, []
    for i in range(2 * n):                           # the second lap lets early elements see later ones
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        if i < n:
            stack.append(i)
    return answer


def next_greater_circular_brute(nums):
    n = len(nums)
    return [next((nums[(i + d) % n] for d in range(1, n) if nums[(i + d) % n] > nums[i]), -1) for i in range(n)]


assert next_greater([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1]
assert next_greater_element([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]
assert next_greater_element([2, 4], [1, 2, 3, 4]) == [3, -1]
assert next_greater_circular([1, 2, 1]) == [2, -1, 2]
assert next_greater_circular([1, 2, 3, 4, 3]) == [2, 3, 4, -1, 4]
assert next_greater_circular([]) == [] and next_greater_circular([5, 5]) == [-1, -1]
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(0, 12))]
    assert next_greater_circular(nums) == next_greater_circular_brute(nums)
print("next greater (linear, subset lookup, circular): all cases pass; circular matches the brute force")

**Complexity:** O(n) time (O(2n) for the circular scan), O(n) space. The same stack, flipped (`>` instead of `<`, or scanning right to left), gives *next smaller*, *previous greater* and *previous smaller* — the building blocks of "largest rectangle in a histogram" (LC 84) and "sum of subarray minimums" (LC 907).

## Problem 6 · Queue using two stacks (LC 232)

**Statement:** implement a FIFO queue with `push`, `pop`, `peek`, `empty` using only stack operations.

**Optimal:** push onto an `in` stack; pop from an `out` stack. When `out` is empty, pour everything from `in` into `out` — that reverses the order, so the oldest element ends up on top. Each element is moved **at most once**, so any sequence of n operations costs O(n): **amortised O(1)** per operation (a single `pop` can be O(n)).

In [ ]:
class TwoStackQueue:
    def __init__(self):
        self._in, self._out = [], []
        self.moves = 0                                   # instrumentation: elements poured from in to out

    def push(self, x):
        self._in.append(x)

    def _refill(self):
        if not self._out:                                # only when out is empty, or FIFO order breaks
            while self._in:
                self._out.append(self._in.pop())
                self.moves += 1

    def pop(self):
        self._refill()
        return self._out.pop()

    def peek(self):
        self._refill()
        return self._out[-1]

    def empty(self):
        return not self._in and not self._out


q = TwoStackQueue()
q.push(1)
q.push(2)
print("peek:", q.peek(), " pop:", q.pop(), " empty:", q.empty())
assert (q.peek(), q.pop(), q.empty()) == (2, 2, True)

for _ in range(200):                                     # random operations vs collections.deque
    q, ref, pushes = TwoStackQueue(), deque(), 0
    for _ in range(40):
        if ref and rng.random() < 0.45:
            assert q.pop() == ref.popleft()
        else:
            x = rng.randint(0, 99)
            q.push(x)
            ref.append(x)
            pushes += 1
        assert q.empty() == (not ref)
    assert q.moves <= pushes                             # every element moved at most once: amortised O(1)
print("TwoStackQueue: 200 random sequences behave like a deque; moves never exceed pushes")

## Problem 7 · Sliding window maximum — the monotonic deque (LC 239)

**Statement:** the maximum of every window of size k. `[1,3,-1,-3,5,3,6,7]`, k=3 → `[3,3,5,5,6,7]`.

**Brute force:** `max` of every window — O(n·k). **Optimal, in plain English:** keep a deque of indices whose values are *decreasing* from front to back. A new value evicts every smaller value at the back — those can never be a maximum again, because the new value is both larger and newer. The front is the current maximum; drop it once it slides out of the window.

In [ ]:
def max_sliding_window_brute(nums, k):
    return [max(nums[i:i + k]) for i in range(len(nums) - k + 1)]


def max_sliding_window(nums, k, trace=None):
    dq, result = deque(), []                    # indices; values decrease from front to back
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:         # smaller (or equal) and older: never a max again
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                      # the front slid out of the window
            dq.popleft()
        if i >= k - 1:
            result.append(nums[dq[0]])          # the front is the max of nums[i-k+1 .. i]
            if trace is not None:
                trace.append((i - k + 1, i, list(dq), nums[dq[0]]))
    return result


for (nums, k), expected in [(([1, 3, -1, -3, 5, 3, 6, 7], 3), [3, 3, 5, 5, 6, 7]), (([1], 1), [1]), (([1, -1], 1), [1, -1]),
                            (([9, 8, 7, 6], 2), [9, 8, 7]), (([4, 4, 4], 2), [4, 4]), (([2, 7, 1], 3), [7])]:
    assert max_sliding_window(nums, k) == expected == max_sliding_window_brute(nums, k), (nums, k)
for _ in range(300):
    nums = [rng.randint(-9, 9) for _ in range(rng.randint(1, 20))]
    k = rng.randint(1, len(nums))
    assert max_sliding_window(nums, k) == max_sliding_window_brute(nums, k)
print("max_sliding_window: edge cases (k = 1, k = n, falling, ties) + 300 random inputs agree with the brute force")

In [ ]:
nums, k, rows = [1, 3, -1, -3, 5, 3, 6, 7], 3, []
max_sliding_window(nums, k, rows)
fig, ax = plt.subplots(figsize=(10, 0.55 * len(rows) + 0.9))
for r, (lo, hi, dq_idx, best) in enumerate(rows):
    y = len(rows) - 1 - r
    for i, v in enumerate(nums):
        in_window, in_deque = lo <= i <= hi, i in dq_idx
        face = COLORS[0] if in_deque else ("#b7d3f6" if in_window else "#f0efec")
        ink = "white" if in_deque else ("#0b0b0b" if in_window else "#898781")
        ax.add_patch(plt.Rectangle((i, y), 0.92, 0.8, facecolor=face, edgecolor="none"))
        ax.text(i + 0.46, y + 0.4, str(v), ha="center", va="center", color=ink, fontsize=10)
    ax.text(len(nums) + 0.3, y + 0.4, f"deque values {[nums[j] for j in dq_idx]} -> max {best}", va="center", fontsize=9)
ax.set_xlim(-0.2, len(nums) + 6)
ax.set_ylim(-0.2, len(rows))
ax.axis("off")
ax.set_title("Sliding window max, k = 3: window in light blue, indices kept in the deque in dark blue")
plt.show()

**Complexity:** O(n) time — each index enters and leaves the deque once; O(k) space. Contrast with [notebook 03](03_hashing_and_sliding_window.ipynb): a *sum* can be updated by subtraction, a *max* cannot — the deque is what makes the max incremental. (A heap also works, in O(n log n).)

## Problem 8 · BFS preview: shortest path in a grid

**Statement:** in a grid of open cells (0) and walls (1), the fewest 4-directional steps from start to goal, or −1. (LC 1091 is the 8-directional version.)

**Why a queue:** BFS explores in rings — every cell at distance d is taken off the queue before any cell at distance d + 1 — so the **first** time BFS reaches a cell, it has found the shortest path. Replace `popleft()` with `pop()` and the same code becomes an iterative DFS, which finds *a* path but not the shortest. Full treatment in [11 · Graphs](11_graphs_bfs_dfs_topo_dijkstra.ipynb).

In [ ]:
def bfs_distances(grid, start):
    """Steps from start to every reachable open cell (BFS)."""
    rows, cols = len(grid), len(grid[0])
    dist = {start: 0}
    queue = deque([start])
    while queue:
        r, c = queue.popleft()                         # FIFO: cells leave in order of distance
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1      # mark when ENQUEUED, so no cell is queued twice
                queue.append((nr, nc))
    return dist


def shortest_path(grid, start, goal):
    return bfs_distances(grid, start).get(goal, -1)


def relaxed_distances(grid, start):
    """Independent check: keep relaxing neighbour distances until nothing changes (slow but obviously correct)."""
    rows, cols = len(grid), len(grid[0])
    dist = {(r, c): float("inf") for r in range(rows) for c in range(cols) if grid[r][c] == 0}
    dist[start], changed = 0, True
    while changed:
        changed = False
        for (r, c) in dist:
            for nb in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if nb in dist and dist[nb] + 1 < dist[(r, c)]:
                    dist[(r, c)], changed = dist[nb] + 1, True
    return {cell: d for cell, d in dist.items() if d < float("inf")}


maze = [[0, 0, 0, 0, 1, 0, 0, 0, 0],
        [1, 1, 0, 1, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 1, 1, 1, 1, 0, 1, 0, 1],
        [0, 0, 0, 0, 1, 0, 0, 0, 0],
        [1, 1, 1, 0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0]]
start, goal = (0, 0), (6, 0)
print("shortest path from", start, "to", goal, "=", shortest_path(maze, start, goal), "steps")
assert shortest_path(maze, start, start) == 0
assert shortest_path([[0, 1], [1, 0]], (0, 0), (1, 1)) == -1           # walled off
for _ in range(100):
    grid = [[int(rng.random() < 0.3) for _ in range(6)] for _ in range(6)]
    grid[0][0] = 0
    assert bfs_distances(grid, (0, 0)) == relaxed_distances(grid, (0, 0))
print("bfs_distances matches brute-force relaxation on 100 random 6x6 grids")

In [ ]:
dist = bfs_distances(maze, start)
rows, cols = len(maze), len(maze[0])
image = np.full((rows, cols), np.nan)
for (r, c), d in dist.items():
    image[r, c] = d
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.imshow(np.array(maze), cmap="Greys", vmin=0, vmax=1.6)                  # walls dark grey
shown = ax.imshow(image, cmap="Blues", vmin=-4, vmax=np.nanmax(image))     # one hue, light = near, dark = far
for (r, c), d in dist.items():
    ax.text(c, r, "S" if (r, c) == start else ("G" if (r, c) == goal else str(d)), ha="center", va="center",
            fontsize=10, fontweight="bold" if (r, c) in (start, goal) else "normal",
            color="white" if d > np.nanmax(image) * 0.6 else "#0b0b0b")
ax.set(title=f"BFS distances from S: the goal G is {dist[goal]} steps away (walls in dark grey)",
       xticks=range(cols), yticks=range(rows))
ax.grid(False)
bar = plt.colorbar(shown, ax=ax, label="steps from S", shrink=0.8)
bar.ax.set_ylim(0, np.nanmax(image))                                     # show only real distances on the scale
plt.show()

**Complexity:** O(rows × cols) time and space — every cell is enqueued at most once and has four neighbours. **Traps:** marking cells visited when *dequeued* instead of enqueued (the same cell gets queued many times); using `list.pop(0)` (quadratic); forgetting bounds checks.

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| matching pairs, nesting, "valid" brackets or tags | stack of openers |
| undo, backspace, "simplify path", "decode string" | stack |
| evaluate RPN or an infix expression | stack of operands (+ a stack of operators for infix) |
| next greater / smaller, "days until", stock span, largest rectangle | monotonic stack of indices |
| O(1) min or max alongside push / pop | store (value, running min) pairs |
| a queue from stacks, "amortised O(1)" | two stacks; pour only when the out-stack is empty |
| process in arrival order, level by level | queue: `collections.deque` |
| shortest path in an unweighted graph or grid | BFS; mark visited when enqueuing |
| maximum / minimum of every window | monotonic deque of indices |
| "requests in the last T seconds", keep the last k items | deque of timestamps / `deque(maxlen=k)` |
| `list.pop(0)` or `list.insert(0, x)` in a loop | replace with `deque.popleft()` / `appendleft()` |

## Try it yourself

**1. Backspace string compare (LC 844).** `#` is a backspace. Are the two typed strings equal? `"ab#c"` vs `"ad#c"` → True; `"a#c"` vs `"b"` → False.

In [ ]:
# Solution — try it yourself first, then run this
def typed(s):
    stack = []
    for ch in s:
        if ch == "#":
            if stack:
                stack.pop()                  # backspace on an empty line does nothing
        else:
            stack.append(ch)
    return "".join(stack)


def backspace_compare(s, t):
    return typed(s) == typed(t)


for (s, t), expected in [(("ab#c", "ad#c"), True), (("ab##", "c#d#"), True), (("a#c", "b"), False), (("###", ""), True), (("a##b", "b"), True)]:
    assert backspace_compare(s, t) == expected, (s, t)
print("backspace_compare: all cases pass (O(n + m) time; a two-pointer scan from the end gets O(1) space)")

**2. A sliding-window rate limiter (LC 933, "number of recent calls").** `ping(t)` records a request at millisecond `t` (non-decreasing) and returns how many requests fall in `[t − 3000, t]`. This is how you would cap calls to an LLM API at N per window ([16_production/02 · Retries, backoff and rate limits](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb)).

In [ ]:
# Solution — try it yourself first, then run this
class RecentCounter:
    def __init__(self, window_ms=3000):
        self.window_ms = window_ms
        self.times = deque()

    def ping(self, t):
        self.times.append(t)
        while self.times[0] < t - self.window_ms:     # drop requests that fell out of the window
            self.times.popleft()
        return len(self.times)


rc = RecentCounter()
answers = [rc.ping(t) for t in [1, 100, 3001, 3002]]
print("pings at 1, 100, 3001, 3002 ->", answers)
assert answers == [1, 2, 3, 3]

limiter, allowed = RecentCounter(window_ms=1000), []   # allow at most 3 requests per second
for t in [0, 200, 400, 600, 1300, 1500]:
    allowed.append(limiter.ping(t) <= 3)
print("allowed:", allowed)
assert allowed == [True, True, True, False, True, True]

**3. Online stock span (LC 901).** `next(price)` returns how many consecutive days, ending today, had a price ≤ today's. Prices `100, 80, 60, 70, 60, 75, 85` → spans `1, 1, 1, 2, 1, 4, 6`.

In [ ]:
# Solution — try it yourself first, then run this
class StockSpanner:
    def __init__(self):
        self.stack = []                               # (price, span), prices strictly decreasing

    def next(self, price):
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]               # absorb the spans this day covers
        self.stack.append((price, span))
        return span


prices = [100, 80, 60, 70, 60, 75, 85]
spanner = StockSpanner()
spans = [spanner.next(p) for p in prices]
print("spans:", spans)
assert spans == [1, 1, 1, 2, 1, 4, 6]
brute = [next((i - j for j in range(i - 1, -1, -1) if prices[j] > prices[i]), i + 1) for i in range(len(prices))]
assert spans == brute
print("StockSpanner matches the brute force (amortised O(1) per call)")

## Say it in an interview

- **30-second answer:** "A stack is LIFO — most recent first — which is exactly what nesting and 'most recent unmatched' need. A queue is FIFO — arrival order — which is what BFS and task processing need. In Python a list is the stack; `collections.deque` is the queue, because `list.pop(0)` shifts every element and is O(n)."
- **Monotonic stack/deque:** "keep candidates in sorted order and discard any that can never be the answer. Each index is pushed and popped once, so it is O(n) overall, even with a nested `while`."
- **Key facts:** list `append`/`pop` O(1) amortised; deque `append`/`appendleft`/`pop`/`popleft` O(1), middle indexing O(n); a queue from two stacks is amortised O(1) per operation; BFS on a grid is O(rows × cols).
- **Follow-ups:** "Thread-safe?" — `queue.Queue` for threads, `asyncio.Queue` for coroutines (both block/await when empty — producer–consumer). "Bounded?" — `deque(maxlen=k)` silently drops the oldest. "Priorities?" — a heap, [10 · Heaps](10_heaps_and_top_k.ipynb).
- **Traps:** popping an empty stack; operand order in RPN; `//` floors (−7 // 2 = −4); BFS marking visited on dequeue; `list.pop(0)` hiding an O(n²).

## Next

- [05 · Linked lists](05_linked_lists.ipynb) — the structure underneath `deque`
- [10 · Heaps and top-k](10_heaps_and_top_k.ipynb) — the priority queue
- [11 · Graphs: BFS, DFS, topological sort, Dijkstra](11_graphs_bfs_dfs_topo_dijkstra.ipynb) — BFS and DFS in full
- GenAI links: [Retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [Async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) (`asyncio.Queue` worker pools) · [Conversation memory and context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb) (`deque(maxlen=k)` as a message buffer)
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 5, "Stacks & queues") · [interview bank](../../ai_interview_prep/index.html)